In [ ]:
import pandas as pd
import numpy as np
import sklearn
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("fraudTrain.csv")

In [ ]:
df.head(2)

## EDA

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

### Data cleaning

In [ ]:
df.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

In [ ]:
# duplicates check

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.isnull().sum()

### EDA

In [ ]:
df['is_fraud'].value_counts()

In [ ]:
sns.countplot(data=df, x='is_fraud')
plt.title('Fraud vs Genuine Transactions')
plt.xlabel('Transaction Type (0=Genuine, 1=Fraud)')
plt.ylabel('Count')
plt.show()

In [ ]:
(df['is_fraud'].value_counts(normalize=True) * 100).round(2)

In [ ]:
plt.figure(figsize=(8,5))

sns.histplot(data=df, x='amt', hue='is_fraud',
             bins=50, log_scale=(False, True))

plt.title('Transaction Amount Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(12,5))

sns.countplot(data=df, x='category', hue='is_fraud')

plt.xticks(rotation=45)
plt.title('Fraud by Transaction Category')
plt.show()

### Correlation

In [ ]:
plt.figure(figsize=(12,8))

sns.heatmap(
    df.select_dtypes(include='number').corr(),
    annot=True,
    cmap='coolwarm'
)

plt.show()

In [ ]:
# Convert date into datetime
df['trans_date_trans_time'] = pd.to_datetime(
    df['trans_date_trans_time']
)

df['dob'] = pd.to_datetime(df['dob'])

# Extract time features
df['transaction_hour'] = df['trans_date_trans_time'].dt.hour

df['transaction_day'] = df['trans_date_trans_time'].dt.day

df['transaction_month'] = df['trans_date_trans_time'].dt.month

df['transaction_weekday'] = df['trans_date_trans_time'].dt.dayofweek

# Calculate customer age at transaction time
df['customer_age'] = (
    (df['trans_date_trans_time'] - df['dob']).dt.days / 365.25
).round(1)

### Calculate distance between customer and merchant

In [ ]:
from math import radians, sin, cos, asin, sqrt

def calculate_distance(lat1, lon1, lat2, lon2):

    lat1, lon1, lat2, lon2 = map(
        radians, [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2

    return 6371 * 2 * asin(sqrt(a))

In [ ]:
df['distance_km'] = df.apply(
    lambda row: calculate_distance(
        row['lat'],
        row['long'],
        row['merch_lat'],
        row['merch_long']
    ),
    axis=1
)

 ### Remove unnecessary columns

In [ ]:
drop_columns = [
    'trans_date_trans_time',
    'cc_num',
    'first',
    'last',
    'street',
    'dob',
    'trans_num',
    'unix_time',
    'Unnamed: 0'
]

df.drop(
    columns=drop_columns,
    inplace=True,
    errors='ignore'
)

### Separate X and y

In [ ]:
X = df.drop('is_fraud', axis=1)

y = df['is_fraud']

In [ ]:
print(X.shape)
print(y.shape)

### Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

### Data Preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

categorical_columns = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numerical_columns = X.select_dtypes(
    include=['number']
).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        min_frequency=20
    ))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numerical_columns),
    ('cat', categorical_pipeline, categorical_columns)
])

### Model Building

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import average_precision_score

model = Pipeline([
    ('preprocessor', clone(preprocessor)),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        tol=1e-3,
        solver='saga',
        random_state=42
    ))
])

model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)

y_prob = model.predict_proba(X_test)[:, 1]

### Model Evaluation

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')

plt.show()

In [ ]:
pr_auc = average_precision_score(y_test, y_prob)

print("PR-AUC:", pr_auc)

### Compare multiple ML models

In [ ]:
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
)

models = {
    'Decision Tree': DecisionTreeClassifier(
        max_depth=12,
        class_weight='balanced',
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )
}


In [ ]:
results = []

for name, classifier in models.items():

    pipeline = Pipeline([
        ('preprocessor', clone(preprocessor)),
        ('classifier', classifier)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    precision = precision_score(y_test, predictions, zero_division=0)
    recall = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    pr_auc = average_precision_score(y_test, probabilities)

    results.append({
        'Model': name,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1,
        'PR-AUC': pr_auc
    })

results_df = pd.DataFrame(results)

results_df


### Model Comparison Visualization

In [ ]:
results_df.set_index('Model')[
    ['Precision', 'Recall', 'F1 Score', 'PR-AUC']
].plot(kind='bar', figsize=(10,5))

plt.title('ML Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.show()

In [ ]:
results_df

### Transformed features

In [ ]:
# Fit a separate copy so this check does not reset the trained model
inspection_preprocessor = clone(preprocessor)
X_train_transformed = inspection_preprocessor.fit_transform(X_train)

print("Shape:", X_train_transformed.shape)
print("Missing values:", np.isnan(X_train_transformed.data).sum()
      if hasattr(X_train_transformed, "data")
      else np.isnan(X_train_transformed).sum())


### Encoding columns

In [ ]:
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

print("Total categorical columns:", len(categorical_columns))
print("Total numerical columns:", len(numerical_columns))

### Feature names

In [ ]:
feature_names = inspection_preprocessor.get_feature_names_out()

print("Total transformed features:", len(feature_names))
print(feature_names[:20])


In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()

print("Total transformed features:", len(feature_names))
print("First 20 transformed features:")

print(feature_names[:20])